In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
 
# Loading data
df = pd.read_csv("data_final_topics.csv")          
 
# Counting comments per post
number = df.groupby("new_post_id").size()
 
print(f"Number of posts: {len(number):,}")
print(f"Average: {number.mean():.2f}, median: {number.median():.0f}, maks: {number.max()}")
 
mean = number.mean()
median = number.median()
 
# Figur with two paneler 
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Arial", "Liberation Serif", "DejaVu Serif"],
    "font.size": 9,
    "axes.linewidth": 0.8,
})
BAR_COLOR = "0.6"          # grey bars (0 = black, 1 = white)

FIG_WIDTH = 6.3          
FIG_HIGHT = 3.2           

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HIGHT))

# Left: zoom in on first 30
limit = 30
ax1.hist(number[number <= limit], bins=np.arange(0.5, limit + 1.5, 1),
         color=BAR_COLOR, edgecolor="white", linewidth=0.5)
ax1.axvline(median, color="black", linestyle="--", linewidth=1.0, label=f"Median = {median:.0f}")
ax1.axvline(mean, color="black", linestyle=":", linewidth=1.4, label=f"Mean = {mean:.1f}")
ax1.set_xlabel("Comments per post")
ax1.set_ylabel("Number of posts")
ax1.set_title(f"Posts with max {limit} comments", fontsize=9, fontweight="bold")
ax1.legend(frameon=False, fontsize=8)

# Right: the entire distribution, logarithmic y-axis, so the tail is visible
ax2.hist(number, bins=np.arange(0.5, number.max() + 1.5, 1),
         color=BAR_COLOR, edgecolor="none")
ax2.set_yscale("log")
ax2.set_xlabel("Comments per post")
ax2.set_ylabel("Number of posts (log-scale)")
ax2.set_title(f"Total distribution (max = {number.max()})", fontsize=9, fontweight="bold")

for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="0.9", linewidth=0.6)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig("comments_per_post.png", dpi=300)
plt.savefig("comments_per_post.pdf")      
plt.show()

In [ ]:
# Number of comments per post (one number per post, corresponds to post_tag == 1)

number = df.groupby("new_post_id").size()
 

print("=== Comments pr. post (post-level) ===")
print(f"Number of posts (Obs): {len(number):,}")
print(f"Average:        {number.mean():.4f}")
print(f"Std. deviation:    {number.std():.4f}")       
print(f"Variance:           {number.var():.4f}")
print(f"Skewness:          {stats.skew(number, bias=True):.4f}")
print(f"Kurtosis:          {stats.kurtosis(number, fisher=False, bias=True):.4f}")
 
# Percentiles
print("\nPercentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    v = np.percentile(number, p, method="averaged_inverted_cdf")
    print(f"  {p:>2}%: {v:g}")
 
print("\nSmallest:", sorted(number)[:4])
print("Largest: ", sorted(number)[-4:])
 
print("\n=== Frequency table (first 10 values) ===")
frequency = number.value_counts().sort_index()
tabel = pd.DataFrame({
    "Freq": frequency,
    "Percent": (frequency / frequency.sum() * 100).round(2),
})
tabel["Cum."] = tabel["Percent"].cumsum().round(2)
print(tabel.head(10))

df["n_comments"] = df.groupby("new_post_id")["new_post_id"].transform("size")
df["share_post"] = (df["n_comments"] > 1).astype(int)
 
print("\n=== Does the comments share post with other? (comment-level) ===")
tab = df["share_post"].value_counts().sort_index()
print(pd.DataFrame({
    "Freq": tab,
    "Percent": (tab / tab.sum() * 100).round(2),
}))
 
print("\n=== Share of posts with more than 1 comment (post-level) ===")
print(f"{(number > 1).mean() * 100:.2f} %")

print(f"\nShare of posts with maximum 42 comments: {(number <= 42).mean() * 100:.2f} %")